#1 GIAI NÉN DỮ LIỆU VÀ KIỂM TRA DUỜNG DẪN DATA

In [ ]:
!pip install ultralytics

In [ ]:
import os
import sys
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO
import zipfile


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Hàm chuyên dụng cho file .zip
def extract_zip_only(zip_path: str, extract_to: str) -> None:
    """
    Chỉ nhận và giải nén file .zip vào Google Drive.
    """
    # Kiểm tra tồn tại
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Không tìm thấy file tại: {zip_path}")

    # Chặn nếu không phải đuôi .zip
    if not zip_path.lower().endswith('.zip'):
        raise ValueError(f"Lỗi: Hàm chỉ nhận file định dạng .zip! File truyền vào: {zip_path}")

    # Kiểm tra file có đúng định dạng zip hợp lệ hay bị hỏng không
    if not zipfile.is_zipfile(zip_path):
        raise zipfile.BadZipFile(f"Lỗi: File '{zip_path}' không phải file zip hợp lệ hoặc bị lỗi tải về dở dang.")

    # Tạo thư mục đích nếu chưa có
    os.makedirs(extract_to, exist_ok=True)

    print(f"Bắt đầu giải nén: {os.path.basename(zip_path)}")
    print(f"vị trí trên Drive: {extract_to}")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Kiểm tra tính toàn vẹn của các file bên trong (CRC checksum)
        corrupt_file = zip_ref.testzip()
        if corrupt_file:
            raise zipfile.BadZipFile(f"File zip bị hỏng dữ liệu tại: {corrupt_file}")

        zip_ref.extractall(extract_to)

    print("Giải nén thành công!")

In [ ]:
# Có thể thay đổi link path theo đường dẫn trên drive của bạn

zip_path = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/rotate_bee640.yolo26.zip"
extract_to = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION"
extract_zip_only(zip_path,extract_to)

### **Trước khi train bạn hãy chỉnh lại đường dẫn trong file yaml sao cho đúng với đường dẫn trên máy bạn**
```python
path: /content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION  #(bạn có thể đổi đường dẫn này dựa trên máy của mình)
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['bee', 'bee ', 'hornet']
```


In [ ]:
yaml_path= r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml"
with open(yaml_path, 'r') as f:
  data = yaml.safe_load(f)
  for key, value in data.items():
    print(f"{key}: {value}")

In [ ]:
def check_paths(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # Duyệt qua các key cần kiểm tra
    for key in ['train', 'val', 'valid', 'test']:
        if key in data:
            path = data[key]
            status = "[PASS] Tồn tại" if os.path.exists(path) else "[ERROR] KHÔNG tồn tại"
            print(f"[{key}]: {path} -> {status}")



path= r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml"
check_paths(path)

#2. kIỂM TRA THIẾT BỊ TRAIN
nếu không phải GPU thì sẽ báo lỗi không thực hiện việc train
- yêu cầu bật chết độ GPU trên google colab

In [ ]:
# Kiểm tra GPU Colab
if not torch.cuda.is_available():
    sys.exit("[ERROR]  Colab đang ở chế độ CPU! Hãy vào Runtime -> Change runtime type -> Chọn GPU (T4)")

device = 0
print(f"[PASS] Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")

#3. TRAIN MODEL YOLO26 BẢN NANO VÀ SMALL

In [ ]:
# Khởi tạo model (Dùng bản chính thức yolov8n.pt hoặc yolov11n.pt)
model = YOLO(r"yolo26s.pt")  # hoặc đường dẫn tuyệt đối đến file

# Định nghĩa đường dẫn bằng Path object để dùng được toán tử /
OUTPUT_PATH = Path("/content/drive/MyDrive/ALL-PROJECT-DATA/BEE-SYSTEMS-PROJECT/BEE-AND-WASP-DECTECTION/MODEL")
YAML_PATH = "/content/drive/MyDrive/ALL-PROJECT-DATA/BEE-SYSTEMS-PROJECT/BEE-AND-WASP-DECTECTION/Data-1/data.yaml"

print("\n--- BẮT ĐẦU HUẤN LUYỆN TRÊN COLAB ---")
results = model.train(
    data=str(YAML_PATH),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,                                     # Chạy trên GPU Colab
    workers=4,
    project=str(OUTPUT_PATH / "runs"),            # Đã hợp lệ nhờ dùng Path
    name="bee_detector",
    exist_ok=True
)
print("\n Huấn luyện hoàn tất!")